[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/geraldmc/irilab2026/blob/main/notebooks/r2/r2-q2/03_sanity_checks_and_gradcam.ipynb)

# R2-Q2 Week 3 — Sanity-check Grad-CAM, then run it on the misclassification set

This notebook does two things in order. First, it runs Adebayo et al. (2018)'s sanity checks on vanilla Grad-CAM applied to the R2-Q1 classifier — the model-parameter randomization check and the data-randomization check — and reports the Spearman rank correlation between trained-model and randomized-model heatmaps at each randomization stage, with ρ < 0.3 at full randomization committed as the passing criterion. Second, once vanilla Grad-CAM has passed the sanity checks, it generates heatmaps for every image in the working set produced by Week 2 and saves them for Week 4's categorization work.

By the end of this notebook you will have:

- A `sanity_check_results.json` recording the Spearman ρ trajectory across randomization stages for both the model-parameter and data-randomization checks, the verdict against the ρ < 0.3 passing criterion, and the methods-section text justifying that the trained model's downstream interpretation rests on a sanity-checked saliency method.
- A `heatmaps/` directory containing one `.npy` file per misclassification — 81 files, each a 7×7 numpy array (pre-upsampling) — plus a `gradcam_metadata.parquet` index that joins the heatmap files back to the working set on `filename`. Week 4 upsamples each heatmap at use time.
- A `shuffled_resnet18.pt` checkpoint — the label-shuffled-trained classifier used for the data-randomization sanity check, saved so the check is reproducible without retraining.
- A diagnostic figure (`sanity_check_trajectory.png`) showing the Spearman ρ curve across randomization stages, so a reader can see the trajectory the verdict was made from, not just the verdict itself.

## Before you run anything: switch to a GPU runtime

This notebook trains a neural network, which needs a more powerful processing 
unit (GPU) to work. By default, Colab gives you a CPU-only runtime — fine for 
last week's notebooks, but it won't work for this one.

To switch:

1. In Colab's menu bar: **Runtime → Change runtime type**.
2. Under "Hardware accelerator," choose **T4 GPU**.
3. Click **Save**.

Wait for the session to come up on the new runtime (a few seconds).
Then run the cells below in order.

**If Colab tells you "no GPUs available right now":** free-tier GPU
access is shared and occasionally unavailable. Wait a few minutes
and try again. If it keeps refusing, tell your mentor — the
upgraded Colab tiers have more reliable GPU access.

In [ ]:
try:
    import irilab2026 as iri
    print(f"OK — irilab2026 {iri.__version__} is installed and all deps importable.")
    print("    → use Pattern 2 in the following Cell:")
    print("      !pip install --force-reinstall --no-deps git+https://github.com/geraldmc/irilab2026.git@main")
except ModuleNotFoundError as e:
    print(f"MISSING — {e.name} not importable.")
    print("    → use Pattern 1 in the following Cell:")
    print("      !pip install git+https://github.com/geraldmc/irilab2026.git@main")

In [ ]:
# Pattern 1 — fresh or partial runtime (installs deps that aren't present yet)
# This skips anything pip already sees as installed. This is ideal for a new environment or when you want
# to be sure everything is up to date.

# !pip install git+https://github.com/geraldmc/irilab2026.git@main

# Pattern 2 — populated runtime (refreshes irilab2026 only, leaves deps alone)
# This is ideal for when you already have the dependencies installed and just want to update irilab2026.

# !pip install --force-reinstall --no-deps git+https://github.com/geraldmc/irilab2026.git@main

### Confirm the GPU is in place

Before going further, confirm Colab attached a GPU. The cell below
prints `GPU detected:` followed by `True` or `False`. If it's `True`,
you're good. If it's `False`, go back to the top of the notebook and
re-do the runtime switch — the next cells will not work without a GPU.

The check is a separate cell from the main setup (below) because it's
fast and recoverable: if the GPU isn't there, you see the problem
right away with no Drive authorization prompt to dismiss first.

In [ ]:
import irilab2026 as iri

gpu_ok = iri.has_gpu()
print(f"GPU detected: {gpu_ok}")

assert gpu_ok, (
    "No GPU detected. Go to the top of the notebook and follow the "
    "runtime-switch instructions, then re-run from Setup Step 1."
)

In [ ]:
# Mount Drive, set up irilab2026 with a GPU requirement, seed everything,
# and point OUTPUT_DIR at the R2-Q1 output directory.
iri.setup(gpu_required=True)
iri.seed_all(42)

OUTPUT_DIR = iri.output_dir("r2_q1")
print(f"Output directory: {OUTPUT_DIR}")

# DEV_MODE switches the notebook between fast-iteration mode (tiny PV
# variant, 2 epochs — runs end to end in ~3 minutes on a T4) and the
# real production run (full PV variant, 10 epochs — ~25 minutes).
#
# Use DEV_MODE = True while you're debugging or making changes. Switch
# to False for the run that produces your real EQ#2 number.
DEV_MODE = True

if DEV_MODE:
    PV_VARIANT = "tiny"
    EPOCHS = 2
    print("DEV_MODE ON: PV variant = 'tiny', epochs = 2")
else:
    PV_VARIANT = "full"
    EPOCHS = 10
    print("DEV_MODE OFF: PV variant = 'full', epochs = 10")

## 2) Load the Week 1 pre-commit

This notebook does not read any specific field from `precommit.json`,
but the file is N01's signed deliverable, and downstream sanity-check
interpretation and Week-4 categorization both depend on it conceptually.
Loading it here is a guardrail: if the file is missing or malformed,
the cell below fails with a clear pointer back to N01 rather than
letting the notebook proceed into Grad-CAM work it can't actually
finish.

The load also light-validates the precommit's top-level structure —
the four keys `taxonomy`, `categorization_procedure`, `sanity_checks`,
and `aggregation_level` must all be present. If any are missing, the
precommit was probably written by an incomplete N01 run, and re-running
N01 to completion is the remedy.

In [ ]:
### 2.1) Load and validate the pre-commit

import json

precommit_path = OUTPUT_DIR / "precommit.json"

try:
    with open(precommit_path) as f:
        precommit = json.load(f)
except FileNotFoundError:
    raise FileNotFoundError(
        f"Could not find {precommit_path}. "
        "This file is produced by 01_pre_commits.ipynb — "
        "run that notebook to completion before running this one."
    ) from None

# Light validation: the four locked top-level keys must all be present.
expected_keys = {
    "taxonomy",
    "categorization_procedure",
    "sanity_checks",
    "aggregation_level",
}
missing = expected_keys - set(precommit.keys())
if missing:
    raise KeyError(
        f"precommit.json is missing top-level keys: {sorted(missing)}. "
        "This usually means N01 didn't complete — "
        "re-run 01_pre_commits.ipynb to regenerate the file."
    )

print(f"Loaded: {precommit_path}")
print(f"  top-level keys: {sorted(precommit.keys())}")

## 3) Load the working set and the classifier

Two loads here. First the working set — the 81-row parquet N02 produced,
containing the PD misclassifications Grad-CAM will run on. Second the
trained classifier — the ResNet-18 weights R2-Q1 N03 trained and saved
as `baseline_resnet18.pt`. Both loads are defensive: if either file is
missing, the cell tells you which notebook to re-run.

Three things happen below the working-set load:

1. **Confirm `baseline_resnet18.pt` exists.** It lives in your R2-Q1
   output directory, not R2-Q2's — Grad-CAM in N03 uses the same model
   R2-Q1 evaluated. If you ran R2-Q1 in a different Drive account or
   ran with a different `OUTPUT_DIR`, the file won't be where this
   cell looks.
2. **Build the architecture and load the weights.** N03 saved only
   the model's state dict (the trained parameter values), not the
   architecture. We rebuild the same ResNet-18 using `iri.build_baseline_model()`,
   read the state dict, take `num_classes` from the saved final layer's
   output dimension, and load the weights. If the saved class count
   doesn't match what `iri.build_baseline_model()` produces, the load
   fails with a size-mismatch error.
3. **Smoke-check with a one-tensor forward pass.** A dummy 224×224
   input goes through the model and we assert the output shape and
   device are correct. Cheap (under a second) and catches a
   "model loaded but doesn't behave right" failure here, not 30 rows
   into Section 5's Grad-CAM loop.

The `model`, `device`, `num_classes`, and `working_set` variables defined
here carry forward into every subsequent section.

In [ ]:
### 3.1) Load and validate the working set

import pandas as pd

working_set_path = OUTPUT_DIR / "working_set.parquet"

try:
    working_set = pd.read_parquet(working_set_path)
except FileNotFoundError:
    raise FileNotFoundError(
        f"Could not find {working_set_path}. "
        "This file is produced by 02_load_and_filter.ipynb — "
        "run that notebook to completion before running this one."
    ) from None

# Schema validation. These nine columns are locked per the N02 handoff —
# if any are missing, N02 produced a file with a different schema than
# this notebook expects, and the safest path is to re-run N02.
expected_columns = {
    "filename", "true_pd_class", "pred_pv_class",
    "true_category", "pred_category", "pred_idx",
    "confidence", "host", "disease",
}
missing = expected_columns - set(working_set.columns)
if missing:
    raise KeyError(
        f"working_set.parquet is missing columns: {sorted(missing)}. "
        "Re-run 02_load_and_filter.ipynb to regenerate the file."
    )

print(f"Loaded: {working_set_path}")
print(f"  rows:        {len(working_set)}")
print(f"  columns:     {sorted(working_set.columns.tolist())}")
print(f"  pred_idx:    range [{working_set['pred_idx'].min()}, {working_set['pred_idx'].max()}], dtype {working_set['pred_idx'].dtype}")
print(f"  confidence:  range [{working_set['confidence'].min():.3f}, {working_set['confidence'].max():.3f}]")

In [ ]:
### 3.1) Load and validate the working set

import pandas as pd

working_set_path = OUTPUT_DIR / "working_set.parquet"

try:
    working_set = pd.read_parquet(working_set_path)
except FileNotFoundError:
    raise FileNotFoundError(
        f"Could not find {working_set_path}. "
        "This file is produced by 02_load_and_filter.ipynb — "
        "run that notebook to completion before running this one."
    ) from None

# Schema validation. These nine columns are locked per the N02 handoff —
# if any are missing, N02 produced a file with a different schema than
# this notebook expects, and the safest path is to re-run N02.
expected_columns = {
    "filename", "true_pd_class", "pred_pv_class",
    "true_category", "pred_category", "pred_idx",
    "confidence", "host", "disease",
}
missing = expected_columns - set(working_set.columns)
if missing:
    raise KeyError(
        f"working_set.parquet is missing columns: {sorted(missing)}. "
        "Re-run 02_load_and_filter.ipynb to regenerate the file."
    )

print(f"Loaded: {working_set_path}")
print(f"  rows:        {len(working_set)}")
print(f"  columns:     {sorted(working_set.columns.tolist())}")
print(f"  pred_idx:    range [{working_set['pred_idx'].min()}, {working_set['pred_idx'].max()}], dtype {working_set['pred_idx'].dtype}")
print(f"  confidence:  range [{working_set['confidence'].min():.3f}, {working_set['confidence'].max():.3f}]")

In [ ]:
### 3.2) Build the ResNet-18 architecture and load the trained weights

import torch

device = torch.device("cuda")

# baseline_resnet18.pt lives in R2-Q1's output directory, not R2-Q2's.
weights_path = iri.output_dir("r2_q1") / "baseline_resnet18.pt"

if not weights_path.exists():
    raise FileNotFoundError(
        f"Could not find {weights_path}. "
        "This file is produced by R2-Q1 Notebook 03 (`03_baseline_classifier.ipynb`). "
        "Run R2-Q1 to completion before running this notebook."
    )

# Read the state dict first. We need num_classes for build_baseline_model,
# and the saved final-layer shape is the authoritative source — deriving
# from the file rather than hardcoding 38 means any future re-training
# with a different class count surfaces here, not downstream.
state_dict = torch.load(weights_path, map_location=device, weights_only=True)
num_classes = state_dict["fc.weight"].shape[0]

# Build the matching architecture, load the weights, switch to eval
# mode (disables dropout, freezes BatchNorm running stats), move to GPU.
model = iri.build_baseline_model(num_classes=num_classes)
model.load_state_dict(state_dict)
model.eval()
model.to(device)

# Cross-check: working_set.pred_idx values must fit within the model's
# output space. If they don't, the working set was built against a
# different model than the one we just loaded.
max_pred_idx = int(working_set["pred_idx"].max())
assert max_pred_idx < num_classes, (
    f"working_set has pred_idx={max_pred_idx}, but the loaded model "
    f"has only {num_classes} output classes. Working set and model "
    "are mismatched — did you regenerate working_set.parquet against "
    "a different model?"
)

print(f"Loaded: {weights_path}")
print(f"  file size:    {weights_path.stat().st_size / 1e6:.1f} MB")
print(f"  num_classes:  {num_classes}")
print(f"  device:       {device}")
print(f"  mode:         {'eval' if not model.training else 'train'}")
print(f"  pred_idx max: {max_pred_idx} (within range)")

In [ ]:
### 3.3) Smoke check — confirm output shape and device

# A single dummy 224×224 RGB input through the model. The point isn't
# to predict anything meaningful — it's to confirm the model produces
# output of the right shape on the right device before Section 4 starts
# wiring Grad-CAM hooks into it.

with torch.no_grad():
    dummy_input = torch.zeros(1, 3, 224, 224, device=device)
    dummy_output = model(dummy_input)

assert dummy_output.shape == (1, num_classes), (
    f"Model output shape is {tuple(dummy_output.shape)}, "
    f"expected (1, {num_classes}). The saved weights may not match the "
    "architecture iri.build_baseline_model() produced."
)
assert dummy_output.device.type == "cuda", (
    f"Model output is on {dummy_output.device}, expected cuda. "
    "model.to(device) may not have applied — re-run cell 3.2."
)

print("Smoke check passed:")
print(f"  output shape:  {tuple(dummy_output.shape)}")
print(f"  output device: {dummy_output.device}")
print(f"  output dtype:  {dummy_output.dtype}")